In [1]:
import pymongo
from typing import List

import sys
sys.path.insert(0, '../')

from src.db_drivers.kv_driver.utils import AbstractKVDatabaseConnection, KVDBConnectionConfig, KeyValueDBInstance
from src.utils import ReturnInfo

/home/dzigen/Desktop/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class MongoKVConnector(AbstractKVDatabaseConnection):

    def __init__(self, config: KVDBConnectionConfig) -> None:
        self.config = config

    def is_open(self) -> bool:
        try:
            self._client.server_info()
            return True
        except pymongo.errors.ServerSelectionTimeoutError as err:
            print(str(err))
            return False

    def open_connection(self):
        self._client = pymongo.MongoClient(f'mongodb://{self.config.host}:{self.config.port}')
        self._collection = self._client[self.config.db_info['db']][self.config.db_info['table']]

    def close_connection(self):
        self._client.close()
    
    def create(self, items: List[KeyValueDBInstance]):
        filtered_items = [item for item in items if self._collection.find_one({'_id': item.id})]
        self._collection.insert_many(filtered_items)

    def read(self, ids: List[str]) -> List[KeyValueDBInstance]:
        items = self._collection.find({"_id": {"$in": ids}})
        items_dict = {item.id: item for item in items}

        sorted_items = []
        for id in ids:
            item = items_dict.get(id, None)
            if item is not None:
                item = KeyValueDBInstance(id=item['_id'], value=item['value'])
            
            sorted_items.append(item)

        return sorted_items

    def update(self, items: List[KVDBConnectionConfig]):
        existig_items = self._collection.find({"_id": {"$in": [item.id for item in items]}})
        
        items_dict = {item.id: item for item in items}
        filtered_items = {items_dict[item.id] for item in existig_items}

        for item in filtered_items:
            self._collection.update_one({'_ids': item.id}, {"$set": { "value": item.value}})

    def delete(self, ids: List[str]):
        self._collection.delete_many({'_id': {"$in": ids}})

    def count_items(self) -> int:
        pass

    def item_exist(self, id: str) -> bool:
        item = self._collection.find_one({'_id': id})
        return item is not None


In [3]:
mongo_config = KVDBConnectionConfig()
mongo_conn = MongoKVConnector(mongo_config)

TypeError: Can't instantiate abstract class MongoKVConnector with abstract methods clear, count_items, is_open, item_exist